# InsightOS Qwen3-8B Orchestration Baseline (Evaluation Only)

This notebook evaluates the untouched `Qwen/Qwen3-8B` checkpoint at immutable revision `b968826d9c46dd6066d109eabc6255188de91218`. It does not load an adapter, train, or dispatch tools. Four-bit NF4 loading is used for Kaggle memory efficiency; record that quantization when comparing any later model.

Upload `qwen3-8b-baseline-bundle-v1.json` from `backend/training/milestone4/` as a private Kaggle dataset named `insightos-qwen3-8b-baseline-v1`, attach it to this notebook, enable a GPU, then run all cells. The bundle contains the finalized 12-case held-out benchmark and SHA-256-bound snapshots of actual InsightOS tool and structured-output contracts. The benchmark is a smoke suite, not a statistically powered certification set; semantic judgments still require human review.

Baseline status remains **pending** until a project owner runs this notebook and reviews its saved report. The notebook has intentionally not been executed here.

In [ ]:
%pip install --quiet "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.45.5" "safetensors==0.5.3" "jsonschema==4.26.0"

import importlib.metadata as metadata
import json
import platform
import sys
import torch

required_packages = ["transformers", "accelerate", "bitsandbytes", "safetensors", "jsonschema"]
package_versions = {}
for package in required_packages:
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError(f"Required package is unavailable: {package}") from error
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled Kaggle GPU is required; no inference was run")
gpu = torch.cuda.get_device_properties(0)
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": gpu.name,
    "gpu_total_memory_bytes": gpu.total_memory,
    "package_versions": package_versions,
}
print(json.dumps(environment, indent=2, sort_keys=True))

The dependency versions above are pinned for this evaluation candidate; Kaggle's CUDA/PyTorch wheel is left untouched and captured in the report. This does not certify compatibility with the earlier 0.6B training session: its checked-in notebook has no execution outputs or package-version record. If installation requests a kernel restart, restart and run all cells.

In [ ]:
from pathlib import Path
import hashlib
from jsonschema import Draft202012Validator

BUNDLE_PATH = Path("/kaggle/input/insightos-qwen3-8b-baseline-v1/qwen3-8b-baseline-bundle-v1.json")
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Attach the baseline bundle Kaggle dataset: {BUNDLE_PATH}")
bundle_bytes = BUNDLE_PATH.read_bytes()
bundle = json.loads(bundle_bytes)
benchmark = bundle["benchmark"]
canonical = lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
sha256 = lambda value: hashlib.sha256(value).hexdigest()
tool_schemas = bundle["tool_schemas"]
contracts = bundle["structured_output_contracts"]
if bundle.get("bundle_version") != "insightos-qwen3-8b-baseline-bundle-v1":
    raise ValueError("Unexpected baseline bundle version")
if bundle["benchmark_sha256"] != sha256(canonical(benchmark)):
    raise ValueError("Benchmark content does not match the bundle hash")
if bundle["benchmark_dataset_sha256"] != sha256(canonical(benchmark["cases"])):
    raise ValueError("Benchmark case dataset hash mismatch")
if bundle["tool_schemas_sha256"] != sha256(canonical(tool_schemas)):
    raise ValueError("InsightOS tool schema hash mismatch")
if bundle["structured_output_contracts_sha256"] != sha256(canonical(contracts)):
    raise ValueError("InsightOS structured-output contract hash mismatch")
if benchmark["tool_schemas_sha256"] != bundle["tool_schemas_sha256"]:
    raise ValueError("Benchmark and bundle use different tool schemas")
if len(benchmark["cases"]) != 12 or len({case["id"] for case in benchmark["cases"]}) != 12:
    raise ValueError("Expected 12 unique independent benchmark cases")
if bundle["model"]["id"] != "Qwen/Qwen3-8B" or bundle["model"]["revision"] != "b968826d9c46dd6066d109eabc6255188de91218":
    raise ValueError("Bundle is not pinned to the approved base model revision")
tool_by_name = {item["function"]["name"]: item["function"]["parameters"] for item in tool_schemas}
if len(tool_by_name) != len(tool_schemas):
    raise ValueError("Tool schema snapshot contains duplicate names")
response_validator = Draft202012Validator(contracts["benchmark_response"])
specialist_validator = Draft202012Validator(contracts["specialist_input"])
print({"benchmark_cases": len(benchmark["cases"]), "registered_tools": len(tool_by_name), "bundle_sha256": sha256(bundle_bytes), "benchmark_sha256": bundle["benchmark_sha256"], "benchmark_file_sha256": bundle["benchmark_file_sha256"], "tool_schemas_sha256": bundle["tool_schemas_sha256"]})

The benchmark's expected behaviors and scoring rules are intentionally excluded from model prompts. Tool schemas are provided as text for proposal-only evaluation; no InsightOS application code, database, or tool executor is imported.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

MODEL_ID = bundle["model"]["id"]
MODEL_REVISION = bundle["model"]["revision"]
MAX_INPUT_TOKENS = 6144
MAX_NEW_TOKENS = 512
SEED = 20261105
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False
)
if not tokenizer.chat_template:
    raise RuntimeError("Pinned Qwen3 tokenizer has no chat template")
template_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Return a JSON object."}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
if not isinstance(template_probe, str) or not template_probe:
    raise RuntimeError("Qwen3 chat-template smoke check failed")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
loaded_revision = getattr(model.config, "_commit_hash", None)
if loaded_revision and loaded_revision != MODEL_REVISION:
    raise RuntimeError(f"Loaded model revision mismatch: {loaded_revision}")
if getattr(model.config, "_name_or_path", MODEL_ID) != MODEL_ID:
    raise RuntimeError("Loaded model identity differs from the pinned base model")
model_identity = {
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "loaded_revision": loaded_revision,
    "model_config_name_or_path": model.config._name_or_path,
    "tokenizer_name_or_path": tokenizer.name_or_path,
    "tokenizer_commit_hash": getattr(tokenizer, "init_kwargs", {}).get("_commit_hash"),
    "adapter_loaded": False,
    "inference_quantization": "4-bit NF4, double quantization, float16 compute",
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "chat_template_sha256": sha256(tokenizer.chat_template.encode("utf-8")),
}
print(json.dumps(model_identity, indent=2, sort_keys=True))

In [ ]:
import math
import re
import statistics
import traceback
from datetime import datetime, timezone

tool_names = set(tool_by_name)
tool_arg_pattern = re.compile(r"\b(?:dataset_id|file_id|analysis_id)\s*=\s*(\d+)")
numeric_pattern = re.compile(r"(?<![A-Za-z])\d+(?:\.\d+)?%?")
execution_claim_pattern = re.compile(r"\b(?:i|we)\s+(?:have\s+)?(?:run|executed|performed|built|created)\b", re.I)
causal_claim_pattern = re.compile(r"\b(?:caused by|the campaign caused|proves? causation|causally attributable)\b", re.I)
engine_tools = {"normalize_file", "run_quality_analysis", "run_eda_analysis", "run_monthly_forecast"}

def prompt_for(case):
    allowed = [tool for tool in tool_schemas if tool["function"]["name"] in case.get("tools", [])]
    payload = {
        "benchmark_case_id": case["id"],
        "user_request": case["prompt"],
        "available_tool_schemas": allowed,
        "mock_tool_result_fixture": case.get("tool_result_fixture"),
    }
    system = (
        "You are the InsightOS analytics orchestrator. Respond with exactly one JSON object "
        "matching the supplied response schema. Tool calls are proposals only and will never "
        "be executed. Never claim a tool ran without a supplied result. Use only supplied "
        "evidence; clarify missing prerequisites, reject unsupported causal claims, and avoid "
        "invented facts. Do not copy or discuss evaluation instructions.\n"
        "RESPONSE_SCHEMA=" + json.dumps(contracts["benchmark_response"], sort_keys=True) + "\n"
        "CASE=" + json.dumps(payload, ensure_ascii=False, sort_keys=True)
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": case["prompt"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

def parse_one_json(text):
    decoder = json.JSONDecoder()
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found in model output")
    value, end = decoder.raw_decode(text[start:])
    if text[start + end:].strip():
        raise ValueError("Unexpected text follows the JSON response")
    return value

def score_case(case, output):
    checks = {}
    errors = []
    try:
        response_validator.validate(output)
        checks["structured_output_valid"] = True
    except Exception as error:
        checks["structured_output_valid"] = False
        errors.append(str(error))
    calls = output.get("tool_calls", []) if isinstance(output, dict) else []
    names = [call.get("name") for call in calls if isinstance(call, dict)]
    checks["tool_names_registered"] = all(name in tool_names for name in names)
    checks["tools_within_case_allowlist"] = all(name in case.get("tools", []) for name in names)
    argument_validity = []
    for call in calls:
        name = call.get("name") if isinstance(call, dict) else None
        arguments = call.get("arguments") if isinstance(call, dict) else None
        if name not in tool_by_name or not isinstance(arguments, dict):
            argument_validity.append(False)
            continue
        try:
            Draft202012Validator(tool_by_name[name]).validate(arguments)
            argument_validity.append(True)
        except Exception:
            argument_validity.append(False)
    checks["tool_arguments_match_actual_schemas"] = all(argument_validity)
    expected_name = case.get("expected_tool_name")
    expected_args = case.get("expected_arguments")
    if expected_name is not None:
        matching = [call for call in calls if isinstance(call, dict) and call.get("name") == expected_name]
        checks["expected_tool_selected"] = bool(matching)
        if expected_args is not None:
            checks["expected_arguments_exact"] = any(call.get("arguments") == expected_args for call in matching)
    response = output.get("response", "") if isinstance(output, dict) else ""
    questions = output.get("clarification_questions", []) if isinstance(output, dict) else []
    if case["id"] in {"intent-dashboard-scope", "tool-arguments-strict-identifiers", "clarify-ambiguous-analysis"}:
        checks["clarification_required"] = output.get("decision") == "clarify" and bool(questions)
    if case["id"] == "tool-arguments-strict-identifiers":
        checks["no_tool_call"] = not calls
    if case["id"] == "plan-profile-quality-analysis":
        indexes = {name: names.index(name) for name in ("get_profile", "run_quality_analysis") if name in names}
        checks["profile_before_quality"] = len(indexes) == 2 and indexes["get_profile"] < indexes["run_quality_analysis"]
    if case["id"] == "specialist-data-delegation":
        handoff = output.get("specialist_handoff")
        try:
            specialist_validator.validate(handoff)
            checks["specialist_handoff_valid"] = True
        except Exception as error:
            checks["specialist_handoff_valid"] = False
            errors.append(str(error))
    if case["id"] == "interpret-tool-result":
        checks["returned_finding_present"] = "customer_id" in response and "2" in response
    if case["id"] == "tool-error-bounded-retry":
        checks["retry_count_at_most_one"] = len([name for name in names if name == "list_files"]) <= 1
        checks["no_success_claim_after_failure"] = re.search(r"\b(?:successfully|completed|done)\b", response, re.I) is None
    if case["id"] == "grounded-profile-response":
        checks["profile_values_present"] = "18" in response and "3" in response
    if case["id"] in {"intent-dashboard-scope", "grounded-profile-response", "resume-without-engine-replay"}:
        checks["no_false_execution_claim"] = execution_claim_pattern.search(response) is None
    if case["id"] == "resume-without-engine-replay":
        checks["no_engine_replay"] = not any(name in engine_tools for name in names)
    if case["id"] in {"clarify-ambiguous-analysis", "reject-unsupported-causal-action"}:
        checks["no_causal_claim"] = causal_claim_pattern.search(response) is None
    if case["id"] == "reject-unsupported-causal-action":
        checks["safe_abstention"] = output.get("decision") == "abstain"
    prompt_numbers = set(numeric_pattern.findall(case["prompt"]))
    fixture_numbers = set(numeric_pattern.findall(json.dumps(case.get("tool_result_fixture", {}))))
    answer_numbers = set(numeric_pattern.findall(response))
    checks["numeric_claims_grounded"] = answer_numbers.issubset(prompt_numbers | fixture_numbers)
    return checks, errors

results = []
torch.cuda.reset_peak_memory_stats()
for case in benchmark["cases"]:
    started = time.perf_counter()
    row = {"case_id": case["id"], "category": case["category"], "human_semantic_review_required": True}
    try:
        prompt = prompt_for(case)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
        input_tokens = int(inputs["input_ids"].shape[1])
        if input_tokens > MAX_INPUT_TOKENS:
            raise ValueError(f"Prompt has {input_tokens} tokens, over configured limit {MAX_INPUT_TOKENS}")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        output_tokens = int(generated.shape[1] - inputs["input_ids"].shape[1])
        text = tokenizer.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        parsed = parse_one_json(text)
        checks, validation_errors = score_case(case, parsed)
        row.update({"status": "completed", "raw_output": text, "parsed_output": parsed, "checks": checks, "validation_errors": validation_errors, "input_tokens": input_tokens, "output_tokens": output_tokens})
    except Exception as error:
        row.update({"status": "failed", "failure_type": type(error).__name__, "failure": str(error), "traceback": traceback.format_exc()})
    row["latency_ms"] = round((time.perf_counter() - started) * 1000, 3)
    results.append(row)
    print(case["id"], row["status"], row["latency_ms"], row.get("checks", {}))

check_names = sorted({name for row in results for name in row.get("checks", {})})
aggregate = {
    "case_count": len(results),
    "completed_cases": sum(row["status"] == "completed" for row in results),
    "failed_cases": sum(row["status"] == "failed" for row in results),
    "check_pass_rates": {
        name: sum(bool(row.get("checks", {}).get(name)) for row in results) / len(results)
        for name in check_names
    },
    "latency_ms_p50": statistics.median(row["latency_ms"] for row in results),
    "latency_ms_p95": sorted(row["latency_ms"] for row in results)[max(0, math.ceil(0.95 * len(results)) - 1)],
    "peak_gpu_memory_allocated_bytes": torch.cuda.max_memory_allocated(),
    "peak_gpu_memory_reserved_bytes": torch.cuda.max_memory_reserved(),
    "human_semantic_scoring_status": "pending_human_review",
}
run_report = {
    "report_version": "insightos-qwen3-8b-baseline-run-v1",
    "baseline_status": "executed_pending_human_review" if any(row["status"] == "completed" for row in results) else "failed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": model_identity,
    "environment": environment,
    "generation_settings": {"do_sample": False, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "max_input_tokens": MAX_INPUT_TOKENS, "tool_execution": False},
    "bundle_sha256": sha256(bundle_bytes),
    "benchmark_sha256": bundle["benchmark_sha256"],
    "benchmark_file_sha256": bundle["benchmark_file_sha256"],
    "benchmark_dataset_sha256": bundle["benchmark_dataset_sha256"],
    "tool_schemas_sha256": bundle["tool_schemas_sha256"],
    "structured_output_contracts_sha256": bundle["structured_output_contracts_sha256"],
    "aggregate": aggregate,
    "per_case_results": results,
}
output_dir = Path("/kaggle/working/insightos-qwen3-8b-baseline")
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / f"baseline-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{hashlib.sha256(str(time.time_ns()).encode()).hexdigest()[:8]}.json"
with report_path.open("x", encoding="utf-8") as output:
    json.dump(run_report, output, ensure_ascii=False, indent=2, sort_keys=True)
    output.write("\n")
print(json.dumps({"report_path": str(report_path), "aggregate": aggregate}, indent=2, sort_keys=True))
print("Download the JSON report from Kaggle's Output panel. No training or tool execution occurred.")

The report contains raw model outputs, per-case automated checks, errors, token counts, latency, peak VRAM, fixed generation settings, model/tokenizer identity, dependency/CUDA details, and all input hashes. Automated numeric and text checks are heuristics; review every case semantically. Download the JSON from Kaggle's Output panel and do not mark the baseline complete until the full report is present and human-reviewed.